In [33]:
import numpy as np
import pandas as pd
import requests

In [34]:
BASE_URL = "https://leaderboard-project1-runai-innovation-thibaut.inference.compute.datascience.ch/api/v1"
TEMPLATE_PATH = "/home/renku/work/input-data/Project 1 - DTB/bioreaction_template.csv"
SUBMISSION_PATH = "submissions/ridge_blend.csv"   # predictions to submit (same rows as the template)
TARGETS = ["vcd", "tcd", "substrate", "byproduct"]


def get_leaderboard() -> pd.DataFrame:
    leaderboard = requests.get(f"{BASE_URL}/leaderboard").json()
    return pd.json_normalize(leaderboard["entries"])


def submit(df: pd.DataFrame, user: str, token: str = "") -> dict:
    submission = {"predictions": df.to_dict(orient="records")}
    response = requests.post(f"{BASE_URL}/submit", json=submission, auth=(user, token))
    if (response.status_code // 100) != 2:
        raise RuntimeError(f"Unexpected error (HTTP {response.status_code}): {response.text}")
    return response.json()["metrics"]

In [35]:
get_leaderboard()

,user_name,rank,attempt,timestamp,metrics.vcd_mape,metrics.tcd_mape,metrics.substrate_mape,metrics.byproduct_mape,metrics.mean_mape
0,Team 04,1,2,2026-11-21T16:16:43.084759,7.151,6.903,7.546,6.969,7.142
1,Team 15,2,2,2026-11-21T15:33:11.666159,6.617,6.515,7.949,8.292,7.343
2,Team 23,3,9,2026-11-21T15:52:24.579771,7.269,7.144,8.346,9.282,8.010
3,Team 18,4,5,2026-11-21T11:20:35.546502,7.193,6.889,8.310,9.707,8.025
4,Team 10,5,16,2026-11-21T16:49:48.933824,7.777,7.838,10.746,10.214,9.144
5,TEMPLATE,6,17,2026-11-21T14:24:40.616352,8.037,7.949,12.023,10.059,9.517


In [36]:
template = pd.read_csv(TEMPLATE_PATH, dtype={"batch_id": str})
df = pd.read_csv(SUBMISSION_PATH, dtype={"batch_id": str})
# Sanity checks: same rows / order as the template, no missing or negative values
assert (df[["batch_id", "elapsed_time"]].values == template[["batch_id", "elapsed_time"]].values).all()
assert df[TARGETS].notna().all().all() and (df[TARGETS] > 0).all().all()
metrics = submit(df[["batch_id", "elapsed_time", *TARGETS]], user="Team 10")
metrics

{'vcd_mape': 7.665,
 'tcd_mape': 7.616,
 'substrate_mape': 8.417,
 'byproduct_mape': 8.777,
 'mean_mape': 8.119}

In [37]:
get_leaderboard()

,user_name,rank,attempt,timestamp,metrics.vcd_mape,metrics.tcd_mape,metrics.substrate_mape,metrics.byproduct_mape,metrics.mean_mape
0,Team 04,1,2,2026-11-21T16:16:43.084759,7.151,6.903,7.546,6.969,7.142
1,Team 15,2,2,2026-11-21T15:33:11.666159,6.617,6.515,7.949,8.292,7.343
2,Team 23,3,9,2026-11-21T15:52:24.579771,7.269,7.144,8.346,9.282,8.010
3,Team 18,4,5,2026-11-21T11:20:35.546502,7.193,6.889,8.310,9.707,8.025
4,Team 10,5,17,2026-11-21T16:52:41.843330,7.665,7.616,8.417,8.777,8.119
5,TEMPLATE,6,17,2026-11-21T14:24:40.616352,8.037,7.949,12.023,10.059,9.517
